[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusicalInformatics/miws26/blob/main/alignment_2/alignment_metrics.ipynb)

In [ ]:
import os
import sys

try:
    import google.colab

    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    # Install python packages
    ! pip install git+https://github.com/CPJKU/partitura.git@develop
    ! pip install parangonar
    ! pip install pymatchmaker==0.4.0

    # To be able to access helper modules and data in the repo for this tutorial
    # (not necessary if the jupyter notebook is run locally instead of google colab)
    if not os.path.exists("./miws26"):
        !git clone https://github.com/MusicalInformatics/miws26.git
    sys.path.insert(0, "./miws26/alignment_1/")
    sys.path.insert(0, "./miws26/alignment_2/")
    MEC_VARIATION_DIR = "./miws26/alignment_1/mec-variation"
else:
    # The notebook is in the alignment_2 folder. We use the code and the data of the alignment_1 folder
    ALIGNMENT_1_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "alignment_1")
    sys.path.insert(0, ALIGNMENT_1_DIR)
    MEC_VARIATION_DIR = os.path.join(ALIGNMENT_1_DIR, "mec-variation")

# Evaluating Music Alignments

How do we know if an alignment is *good*? It depends on what the alignment is. In this notebook we look at two kinds of metrics:

* **Note-level metrics**: the alignment says which note of the score corresponds to which note of the performance. It is evaluated by comparing the pairs of notes with a ground truth, using **precision, recall and F-score**. This is the metric used, e.g., for the score-to-performance alignments of the ASAP dataset (Peter et al., *Automatic Note-Level Score-to-Performance Alignments in the ASAP Dataset*, TISMIR 2023).
* **Time-level metrics**: the alignment says where in the score the performance is at each moment in time. It is evaluated by the **error in beats** of the reported position at the time of each performed onset. This is the usual way of evaluating score following (online alignment), but it applies to any alignment that maps performance time to score position.

We use the `mec-variation` dataset, with several performances of the same score and ground truth alignments. The F-score code is in [Parangonar](https://github.com/sildater/parangonar) (`parangonar.evaluate`; the same function is in the `helper.py` of `alignment_1`), and the time-level metrics are in [Matchmaker](https://github.com/pymatchmaker/matchmaker) (`matchmaker.utils.eval`).

In [ ]:
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import partitura as pt
from IPython.display import display
from matchmaker.utils.eval import evaluate_alignment
from parangonar.evaluate import fscore_alignments

from helper import show_figure
# The plots and the code to run and evaluate the alignments are in metrics_utils.py
from metrics_utils import (
    NOTE_LEVEL_METHODS,
    accuracy_curves_figure,
    alignment_to_sets,
    beat_errors,
    beat_errors_figure,
    compute_alignment,
    error_distribution_figure,
    evaluate_note_alignments,
    evaluate_online,
    evaluate_online_followers,
    fscore_table,
    interactive_note_alignment,
    interactive_online_comparison,
    interactive_perturbed_metrics,
    load_ground_truth,
    make_performance_with_errors,
    paths_figure,
    plot_note_alignment,
    plot_note_alignment_results,
    plot_online_results,
    run_followers,
    smoothness_figure,
    tp_fp_fn_figure,
)

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format ='retina'

MUSICXML_DIR = os.path.join(MEC_VARIATION_DIR, "scores", "musicxml")
MATCH_DIR = os.path.join(MEC_VARIATION_DIR, "alignments", "match")
MIDI_DIR = os.path.join(MEC_VARIATION_DIR, "performances", "midi")

## Note-level metrics

A note-level alignment is a list of dictionaries, each with a `label`:

* `match`: a score note (`score_id`) was played as a performed note (`performance_id`).
* `deletion`: a score note was **not played**.
* `insertion`: a performed note is **not in the score** (e.g., a wrong note).

The `.match` file contains a performance and its alignment. (As in `Symbolic_Music_Alignment`, we unfold the score so that the note ids match the alignment.)

In [ ]:
score_fn = os.path.join(MUSICXML_DIR, "short.musicxml")
score = pt.load_score(score_fn)
# The alignments of this dataset refer to the "unfolded" score (i.e., with the repeats written out)
score = pt.score.unfold_part_maximal(score[0])
snote_array = score.note_array()

match_fn = os.path.join(MATCH_DIR, "short1.match")
performance, gt_alignment = pt.load_match(
    match_fn, pedal_threshold=127, first_note_at_zero=True
)
pnote_array = performance.note_array()

print(f"The score has {len(snote_array)} notes")
print(f"The performance has {len(pnote_array)} notes")
gt_alignment[:3]

In [ ]:
from collections import Counter

Counter(a["label"] for a in gt_alignment)

Most notes are matches, and there is one insertion:

In [ ]:
for a in gt_alignment:
    if a["label"] != "match":
        print(a)

### Visualizing alignments

The function `plot_note_alignment` (in `metrics_utils.py`) plots the piano roll of the performance (top) and of the score (bottom), with a line between aligned notes. Notes without a partner (deletions in the score, insertions in the performance) are orange. Given a prediction *and* a ground truth, it colors the pairs by whether they are correct (used later).

In [ ]:
show_figure(
    plot_note_alignment(snote_array, pnote_array, gt_alignment, title="Ground truth alignment")
)

Let's zoom in on a few beats (can you find the insertion?):

In [ ]:
interactive_note_alignment(snote_array, pnote_array, {"ground truth": gt_alignment})

### Predicted alignments

To have something to evaluate, we compute alignments with the offline methods from `Symbolic_Music_Alignment`: DTW on features of the score and the performance, followed by a greedy matching of the notes. We compare a **linear** alignment (constant tempo, no actual alignment), **DTW on pitch classes**, and **DTW on piano rolls**.

In [ ]:
METHODS = NOTE_LEVEL_METHODS
predictions = {m: compute_alignment(m, score, performance) for m in METHODS}

for m, pred in predictions.items():
    print(m, dict(Counter(a["label"] for a in pred)))

### True positives, false positives and false negatives

We compare the *set of pairs* of the prediction with the one of the ground truth:

* **True positives (TP)**: pairs in both the prediction and the ground truth (green).
* **False positives (FP)**: pairs only in the prediction, i.e., wrong pairs (red).
* **False negatives (FN)**: pairs only in the ground truth, i.e., missed pairs (blue, dashed).

A pair is either correct or wrong, no matter how far the predicted partner is from the true one.

In [ ]:
show_figure(
    plot_note_alignment(
        snote_array, pnote_array, predictions["DTW (piano roll)"], gt_alignment,
        title="DTW (piano roll) vs. ground truth",
    )
)

In [ ]:
interactive_note_alignment(snote_array, pnote_array, predictions, gt_alignment)

### Precision, recall and F-score

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN} \qquad \text{F-score} = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

*Precision*: which proportion of the predicted pairs is correct? *Recall*: which proportion of the ground truth pairs was found? The F-score (harmonic mean) is high only if both are. Step by step, for the matches of the DTW (piano roll) alignment:

In [ ]:
pred_matches, pred_deletions, pred_insertions = alignment_to_sets(predictions["DTW (piano roll)"])
gt_matches, gt_deletions, gt_insertions = alignment_to_sets(gt_alignment)

true_positives = pred_matches & gt_matches
false_positives = pred_matches - gt_matches
false_negatives = gt_matches - pred_matches

n_tp, n_fp, n_fn = len(true_positives), len(false_positives), len(false_negatives)
precision = n_tp / (n_tp + n_fp)
recall = n_tp / (n_tp + n_fn)
f_score = 2 * precision * recall / (precision + recall)

print(f"TP = {n_tp}, FP = {n_fp}, FN = {n_fn}")
print(f"Precision = {precision:.4f}, Recall = {recall:.4f}, F-score = {f_score:.4f}")

The bars show the pairs of the ground truth (TP + FN) and of the prediction (TP + FP):

In [ ]:
show_figure(
    tp_fp_fn_figure(
        n_tp, n_fp, n_fn,
        title=f"Precision = TP / prediction = {precision:.3f},   Recall = TP / ground truth = {recall:.3f}",
    )
)

`parangonar.evaluate.fscore_alignments` does this; `types` selects the labels to evaluate:

In [ ]:
precision_pa, recall_pa, f_score_pa = fscore_alignments(
    prediction=predictions["DTW (piano roll)"],
    ground_truth=gt_alignment,
    types=["match"],
)
print(f"Precision = {precision_pa:.4f}, Recall = {recall_pa:.4f}, F-score = {f_score_pa:.4f}")
assert np.allclose([precision, recall, f_score], [precision_pa, recall_pa, f_score_pa])

### What counts as a correct pair?

A predicted item is correct only if the ground truth has an item that is **exactly equal** (same `label`, `score_id` and `performance_id`). A toy example, where the score note `s5` was not played (deletion) and the performed note `p4` is not in the score (insertion):

In [ ]:
toy_gt = [
    {"label": "match", "score_id": "s1", "performance_id": "p1"},
    {"label": "match", "score_id": "s2", "performance_id": "p2"},
    {"label": "match", "score_id": "s3", "performance_id": "p3"},
    {"label": "insertion", "performance_id": "p4"},
    {"label": "deletion", "score_id": "s5"},
]
# A prediction that makes some errors
toy_prediction = [
    {"label": "match", "score_id": "s1", "performance_id": "p1"},  # correct
    {"label": "match", "score_id": "s2", "performance_id": "p3"},  # wrong partner
    {"label": "match", "score_id": "s3", "performance_id": "p2"},  # wrong partner
    {"label": "match", "score_id": "s5", "performance_id": "p4"},  # the wrong note was matched
]

for label in ["match", "insertion", "deletion"]:
    p, r, f, n_pred, n_gt = fscore_alignments(toy_prediction, toy_gt, [label], return_numbers=True)
    print(f"{label:10s} predicted: {n_pred}, ground truth: {n_gt} -> P = {p:.2f}, R = {r:.2f}, F = {f:.2f}")

* Only 1 of the 4 predicted matches is correct (P = 0.25), out of 3 in the ground truth (R = 0.33).
* Nothing was predicted for `insertion` and `deletion`, but the ground truth has one of each: recall is 0 (and the precision is defined as 0).
* If there is nothing in the prediction **and** in the ground truth for a label, the F-score is defined as 1.

### F-score per type

The F-score is usually reported for each label separately (and sometimes for all together, `types=["match", "insertion", "deletion"]`). `match` is the most important since most notes are matches, but `insertion` and `deletion` matter for applications that detect performance errors. They are rare, so their F-scores change a lot with single errors.

In [ ]:
for m in METHODS:
    print(m)
    display(fscore_table(predictions[m], gt_alignment).round(3))

The `all` F-score is dominated by the (many) matches. Note that the insertion precision of the DTW alignments is low even with a recall of 1: they predict many more insertions and deletions than there are in the ground truth.

### Effect of different errors

Let's simulate two kinds of errors on the ground truth:

* **Wrong matches**: some matches are given a wrong partner (adds FPs and FNs to `match`).
* **Missed matches**: some matches are replaced by a `deletion` and an `insertion` (reduces the `match` recall, but adds no wrong matches).

In [ ]:
interactive_perturbed_metrics(gt_alignment)

### Comparing methods and performances

Finally, all three methods on all the short performances of the dataset (two fast, one slow, some with insertions and deletions):

In [ ]:
performance_names = ["short1", "short2", "short_fast1", "short_fast2", "short_slow1"]

results_df = evaluate_note_alignments(score, performance_names, MEC_VARIATION_DIR)
results_df.set_index(["performance", "method"]).round(3)

In [ ]:
show_figure(plot_note_alignment_results(results_df, performance_names))

results_df.groupby("method")[["match precision", "match recall", "match F-score"]].mean().loc[METHODS].round(3)

**Questions**:

* Which method is best? Is it the best on every performance? Why does the linear alignment work well for some of them?
* The F-score is all-or-nothing for each pair, so it says nothing about *how far* a wrong pair is, or about *timing*. What kind of metric would capture that?

## Time-level metrics

Time-level metrics evaluate the **position in the score (in beats) at each moment of the performance**. The alignment is a path of (performance time, score position) pairs, and the metrics measure *how far* the position is from the ground truth. They are the standard for score following, where an algorithm reports the position while the performance is being played, and they work for any alignment that maps time to position.

Matchmaker includes several score followers for MIDI performances, and the evaluation code. We use four of them:

* `pthmm`: the `PitchHMM` from the `score_follower` notebook,
* `arzt`, `dixon`: online time warping (event-level versions of Arzt and Widmer, and Dixon),
* `outerhmm`: an HMM with an outer-product observation model.

In [ ]:
performance_fn = os.path.join(MIDI_DIR, "short1.mid")
paths = run_followers(performance_fn, score_fn)

print("Alignment path of PitchHMM (performance time in s, position in beats):")
print(paths["PitchHMM"][:, :5])

### Ground truth

The ground truth is the **score position (in beats) at the time of each performed onset**, which we get from the `.match` file with `resolve_gt`.

In [ ]:
gt_perf_sec, gt_score_beat = load_ground_truth("short1", snote_array, MEC_VARIATION_DIR)
print(f"{len(gt_perf_sec)} ground truth onsets")
pd.DataFrame({"performance time (s)": gt_perf_sec, "score position (beats)": gt_score_beat}).head()

In [ ]:
show_figure(paths_figure(paths, gt_perf_sec, gt_score_beat))

On a clean performance all the followers are almost perfect. To see the metrics at work, we add **errors to the performance** (wrong pitches and random extra notes). The true position in the score does not change, so the same ground truth applies.

In [ ]:
noisy_fn = make_performance_with_errors("short1", fraction_errors=0.1, mec_variation_dir=MEC_VARIATION_DIR)
noisy_paths = run_followers(noisy_fn, score_fn)

show_figure(paths_figure(noisy_paths, gt_perf_sec, gt_score_beat, zoom=(0, 6), title="Performance with errors"))

### Beat error

For each ground truth onset (performance time $t_i$, true position $b_i$) we look up the position that the follower had reported at that time, $\hat{b}_i$ (`transfer_positions`). The **beat error** is $e_i = b_i - \hat{b}_i$: positive if the follower is *behind*, negative if it is *ahead*. In the plot (Outer-product HMM), the red segments are the errors:

In [ ]:
show_figure(beat_errors_figure(noisy_paths["Outer-product HMM"], gt_perf_sec, gt_score_beat))

Matchmaker summarizes the errors with `evaluate_alignment`. By hand:

In [ ]:
path = noisy_paths["Outer-product HMM"]
predicted, errors = beat_errors(path, gt_perf_sec, gt_score_beat)

print(f"mean absolute error:   {np.nanmean(np.abs(errors)):.4f} beats")
print(f"median absolute error: {np.nanmedian(np.abs(errors)):.4f} beats")
print(f"std of absolute error: {np.nanstd(np.abs(errors)):.4f} beats")

results = evaluate_alignment(
    aligned_score_beats=path[1],
    aligned_perf_sec=path[0],
    gt_score_beats=gt_score_beat,
    gt_perf_sec=gt_perf_sec,
)
assert np.isclose(results["mean_b"], np.nanmean(np.abs(errors)), atol=1e-4)
pd.Series(results, name="Outer-product HMM").to_frame()

* `mean_b`, `median_b`, `std_b`: statistics of the **absolute** beat error. A small median with a large mean means that the follower is mostly right, but sometimes very wrong.
* `skewness`, `kurtosis`: shape of the distribution of the **signed** error (skewed: tends to be late or early; large kurtosis: a few large outliers). `NaN` if there are no errors.
* `accuracy_<tolerance>b`: proportion of onsets with an absolute error of at most 0.05, 0.1, 0.3, 0.5, 1 or 2 beats.
* `smoothness`: smoothness of the reported trajectory (below).

In [ ]:
show_figure(error_distribution_figure(noisy_paths, gt_perf_sec, gt_score_beat, histogram_of="Outer-product HMM"))

Most errors are exactly 0, and a few are large (note the log scale).

### Accuracy at different tolerances

Instead of a mean, we can ask for which proportion of the onsets the follower is within a tolerance. The right tolerance depends on the application (a couple of beats for page turning, much less for automatic accompaniment). The curves are the cumulative distribution of the absolute error, evaluated at Matchmaker's tolerances:

In [ ]:
show_figure(accuracy_curves_figure(noisy_paths, gt_perf_sec, gt_score_beat))

A better follower has a higher curve.

### Smoothness

A follower can be accurate on average and still be useless for accompaniment or page turning if its position jumps around. Matchmaker measures this with the **spectral arc length** (SPARC) of the trajectory:

1. the reported position is resampled at 50 Hz (holding the last report), and its **speed** is computed;
2. we take the magnitude of the Fourier transform of the speed (normalized to a maximum of 1);
3. the smoothness is the **negative arc length** of this spectrum. A smooth trajectory has a simple spectrum (a short curve); jumps add a lot of high-frequency content (a long curve). Values closer to 0 are smoother.

It is computed in windows of 30 s and averaged (our performance fits in one window), and depends on the tempo, so compare it only between followers on the *same* performance.

In [ ]:
smooth_and_jerky = {name: noisy_paths[name] for name in ["PitchHMM", "Outer-product HMM"]}
show_figure(smoothness_figure(smooth_and_jerky, gt_perf_sec))

### All metrics together

In [ ]:
pd.DataFrame(
    {name: evaluate_online(p, gt_perf_sec, gt_score_beat) for name, p in noisy_paths.items()}
).round(3)

### Robustness to errors

Change the performance and the fraction of errors, and compare the followers:

In [ ]:
interactive_online_comparison(performance_names, score_fn, snote_array, MEC_VARIATION_DIR)

### Comparing followers on all performances

All short performances, clean and with 10% errors:

In [ ]:
online_df = evaluate_online_followers(performance_names, score_fn, snote_array, MEC_VARIATION_DIR)

metrics = ["mean_b", "median_b", "accuracy_0.05b", "accuracy_0.3b", "accuracy_1b", "smoothness"]
online_df.groupby(["condition", "follower"])[metrics].mean().loc[["clean", "10% errors"]].round(3)

In [ ]:
show_figure(plot_online_results(online_df, performance_names))

**Questions**:

* On the clean performances, which metric separates the followers best? How do the errors change the picture?
* Which follower would you choose for page turning? And for automatic accompaniment?

## Summary

| | Note-level | Time-level |
|---|---|---|
| What is evaluated | pairs (score note, performed note) | score position (beats) at each performed onset |
| Ground truth | note alignment (`match`, `insertion`, `deletion`) | position of each performed onset |
| Error | a pair is right or wrong | *how far* the position is (beat error) |
| Summary | precision, recall, F-score per label | mean/median/std error, accuracy at tolerances, skewness/kurtosis, smoothness |

Time-level metrics tell us nothing about wrong or missing notes, and note-level metrics tell us nothing about timing, so both are often reported. (The time-level metrics here use a simulated stream, so they do not include the processing delay of a real-time system.)